In [49]:
import math
import random
import numpy
import torch
import torch.nn as nn
from torch.nn import functional as F

In [50]:
batch_size = 32
block_size = 128
n_embd = 128
n_head = 4
epochs = 100000
learning_rate = 3e-4
n_layers = 4
dropout = 0.2
device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f"using device: {device}")

using device: mps


In [51]:
with open('data/grabbed_nietzsche.txt', 'r', encoding='utf-8') as f:
    nietzshe = f.read()

In [52]:
len(nietzshe)

5812740

In [53]:
characters = sorted(set(nietzshe))
vocab_size = len(characters)
print(''.join(characters))
print(vocab_size)


 !"&'()*,-./0123456789:;<=>?ABCDEFGHIJKLMNOPQRSTUVWXYZ[\]_abcdefghijklmnopqrstuvwxyz}§ÀÄÆÈÉÊÏÜàâäæçèéêîïôöùûüŒœΚΣΤάέήίαβγδεζηθικλμνξοπρςστυφχψωόύώἀἃἄἆἈἐἑἒἔἰὀὖὰὲὶὸᾶ᾽ῑῡῢῶ‐–—‘’“”…
177


In [54]:
stoi = {s:i for i, s in enumerate(characters)}
itos = {i:s for s, i in stoi.items()}
print(stoi)
print(itos)                                     # the encoders we are using are character to token encoders

encode = lambda s: [stoi[c] for c in s]
decode = lambda s: ''.join([itos[c] for c in s])
encode('hi there')
decode(encode("hi there"))


{'\n': 0, ' ': 1, '!': 2, '"': 3, '&': 4, "'": 5, '(': 6, ')': 7, '*': 8, ',': 9, '-': 10, '.': 11, '/': 12, '0': 13, '1': 14, '2': 15, '3': 16, '4': 17, '5': 18, '6': 19, '7': 20, '8': 21, '9': 22, ':': 23, ';': 24, '<': 25, '=': 26, '>': 27, '?': 28, 'A': 29, 'B': 30, 'C': 31, 'D': 32, 'E': 33, 'F': 34, 'G': 35, 'H': 36, 'I': 37, 'J': 38, 'K': 39, 'L': 40, 'M': 41, 'N': 42, 'O': 43, 'P': 44, 'Q': 45, 'R': 46, 'S': 47, 'T': 48, 'U': 49, 'V': 50, 'W': 51, 'X': 52, 'Y': 53, 'Z': 54, '[': 55, '\\': 56, ']': 57, '_': 58, 'a': 59, 'b': 60, 'c': 61, 'd': 62, 'e': 63, 'f': 64, 'g': 65, 'h': 66, 'i': 67, 'j': 68, 'k': 69, 'l': 70, 'm': 71, 'n': 72, 'o': 73, 'p': 74, 'q': 75, 'r': 76, 's': 77, 't': 78, 'u': 79, 'v': 80, 'w': 81, 'x': 82, 'y': 83, 'z': 84, '}': 85, '§': 86, 'À': 87, 'Ä': 88, 'Æ': 89, 'È': 90, 'É': 91, 'Ê': 92, 'Ï': 93, 'Ü': 94, 'à': 95, 'â': 96, 'ä': 97, 'æ': 98, 'ç': 99, 'è': 100, 'é': 101, 'ê': 102, 'î': 103, 'ï': 104, 'ô': 105, 'ö': 106, 'ù': 107, 'û': 108, 'ü': 109, 'Œ': 11

'hi there'

In [55]:
tokenized_nietzshe = torch.tensor(encode(nietzshe), dtype=torch.long)       # encode the entirety of nietzshe into tokens
                                                                            

tokenized_nietzshe[:1000]

tensor([ 48,  36,  33,   1,  48,  51,  37,  40,  37,  35,  36,  48,   1,  43,
         34,   1,  48,  36,  33,   1,  37,  32,  43,  40,  47,   0,   0,  30,
         53,   0,   0,  34,  46,  37,  33,  32,  46,  37,  31,  36,   1,  42,
         37,  33,  48,  54,  47,  31,  36,  33,   0,   0,  43,  76,   9,   1,
         36,  73,  81,   1,  78,  73,   1,  44,  66,  67,  70,  73,  77,  73,
         74,  66,  67,  77,  63,   1,  81,  67,  78,  66,   1,  78,  66,  63,
          1,  36,  59,  71,  71,  63,  76,   0,   0,   0,  48,  36,  33,   1,
         29,  42,  48,  37,  31,  36,  46,  37,  47,  48,   0,   0,  58,  42,
         43,  48,  33,  47,   1,  48,  43,   1,  54,  29,  46,  29,  48,  36,
         49,  47,  48,  46,  29,   9,   1,  29,  42,  32,   1,  33,  48,  33,
         46,  42,  29,  40,   1,  46,  33,  31,  49,  46,  46,  33,  42,  31,
         33,  58,   0,   0,   0,  48,  46,  29,  42,  47,  40,  29,  48,  33,
         32,   1,  30,  53,   0,   0,  29,  42,  48,  36,  43,  

In [56]:
# split into train and test data
n = int(len(tokenized_nietzshe)*.9)
train_nietzshe = tokenized_nietzshe[:n]
test_nietzshe = tokenized_nietzshe[n:]

print(train_nietzshe.shape, test_nietzshe.shape)

torch.Size([5231466]) torch.Size([581274])


In [57]:
train_nietzshe[:block_size+1]

tensor([48, 36, 33,  1, 48, 51, 37, 40, 37, 35, 36, 48,  1, 43, 34,  1, 48, 36,
        33,  1, 37, 32, 43, 40, 47,  0,  0, 30, 53,  0,  0, 34, 46, 37, 33, 32,
        46, 37, 31, 36,  1, 42, 37, 33, 48, 54, 47, 31, 36, 33,  0,  0, 43, 76,
         9,  1, 36, 73, 81,  1, 78, 73,  1, 44, 66, 67, 70, 73, 77, 73, 74, 66,
        67, 77, 63,  1, 81, 67, 78, 66,  1, 78, 66, 63,  1, 36, 59, 71, 71, 63,
        76,  0,  0,  0, 48, 36, 33,  1, 29, 42, 48, 37, 31, 36, 46, 37, 47, 48,
         0,  0, 58, 42, 43, 48, 33, 47,  1, 48, 43,  1, 54, 29, 46, 29, 48, 36,
        49, 47, 48])

In [58]:
v = torch.randint(n-block_size, (batch_size, ))     # how to sample random values from the trainset
for i in v:
    print(i.item())

4186779
5142961
530468
3740093
3992485
4657096
3359960
3863889
2778961
4458462
3402997
1429691
2031629
1783344
1260637
4225964
3974154
4031693
2552427
2417024
5066570
731815
2320913
2986698
87227
3455838
4960251
176252
417012
1926181
5182926
1747557


In [59]:

def generate_batch(type=None):
    xb = []
    yb = []

    if type == 'test':
        data = test_nietzshe
    else:
        data = train_nietzshe

    n = len(data)
    start_vals = torch.randint(n-block_size, (batch_size, ))     # how to sample random values from the trainset 
    for start_val in start_vals:
        start_val = start_val.item()
        xb.append(data[start_val:start_val+block_size])
        yb.append(data[start_val+1:start_val+block_size+1])
    
    xb = torch.stack(xb)
    yb = torch.stack(yb)
    return xb.to(device), yb.to(device)                          # move batches onto the active device

xb, yb = generate_batch()

print(xb, yb)

tensor([[  1,  77,  74,  ...,  67,  78,  72],
        [ 79,  78,   1,  ...,  48,  73,   1],
        [ 78,  73,   1,  ..., 175,   0,   0],
        ...,
        [ 33,   1,  31,  ...,  78,  76,  79],
        [ 59,  78,  67,  ...,  67,  78,  77],
        [ 81,  67,  77,  ...,  63,   9,   1]], device='mps:0') tensor([[ 77,  74,  63,  ...,  78,  72,  63],
        [ 78,   1,  73,  ...,  73,   1,  66],
        [ 73,   1,  65,  ...,   0,   0, 174],
        ...,
        [  1,  31,  43,  ...,  76,  79,  61],
        [ 78,  67,  73,  ...,  78,  77,  61],
        [ 67,  77,  66,  ...,   9,   1,  59]], device='mps:0')


In [60]:
class SelfAttention(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size)
        self.query = nn.Linear(n_embd, head_size)
        self.value = nn.Linear(n_embd, head_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, xb):                                              # (B, T, C)
        k = self.key(xb)                                                # (B, T, head_size=n_embd/n_head)
        q = self.query(xb)                                              # (B, T, head_size)
        v = self.value(xb)                                              # (B, T, head_size)

        B, T, C = k.shape

        weights = q @ k.transpose(-2, -1) / C**0.5                      # (B, T, C) @ (B, C, T) => (B, T, T)
                                                                        # divided by n_embd root to normalize the values, bring them to one
        mask = torch.tril(torch.ones(T, T, device=xb.device))          # triangular mask, built on the same device as the input
        weights = weights.masked_fill(mask == 0, float('-inf'))      
        weights = F.softmax(weights, dim=-1)                            # softmax to get probabilities (B, T, T)
        weights = self.dropout(weights)

        out = weights @ v                                               # (B, T, T) @ (B, T, head_size) => (B, T, head_size)
        return out
      

In [61]:
class MultipleSelfAttention(nn.Module):
    def __init__(self, n_head, n_embd):
        super().__init__()
        self.heads = nn.ModuleList([SelfAttention(n_embd // n_head) for i in range(n_head)])
        self.lin_proj = nn.Linear(n_embd, n_embd)

    def forward(self, xb):
        full_attention = []

        for head in self.heads: 
            full_attention.append(head(xb))                             # (B, T, head_size)
        
        full_attention = torch.cat(full_attention, dim=-1)              # add n_head of (B, T, head_size) along last dimension = (B, T, C)
        full_attention = self.lin_proj(full_attention)                  # linear projection of (B, T, C) to get (B, T, C)

        return full_attention


In [69]:
class CompleteSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        self.head_size = n_embd // n_head
        self.key = nn.Linear(n_embd, n_embd)
        self.query = nn.Linear(n_embd, n_embd)
        self.value = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)
        self.lin_proj = nn.Linear(n_embd, n_embd)
        self.register_buffer('mask', torch.tril(torch.ones(block_size, block_size, device=xb.device)))

        


    def forward(self, xb):
        B, T, C = xb.shape

        k = self.key(xb).reshape((B, T, n_head, self.head_size)).transpose(-2, -3)                  # (B, n_head, T, head_size) 
        q = self.query(xb).reshape((B, T, n_head, self.head_size)).transpose(-2, -3)                                  
        v = self.value(xb).reshape((B, T, n_head, self.head_size)).transpose(-2, -3)    

        weights = q @ k.transpose(-2, -1) / self.head_size**0.5                                                  # (B, n_head, T, head_size) @ (B, n_head, head_size, T) 
        weights = weights.masked_fill(self.mask[:T, :T] == 0, float('-inf'))
        weights = F.softmax(weights, dim=-1)
        weights = self.dropout(weights)

        out = (weights @ v).transpose(-2, -3).reshape((B, T, C))
        out = self.lin_proj(out)

        return out




In [70]:
class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.ff = nn.Sequential(
            nn.Linear(n_embd, 4*n_embd),
            nn.GELU(),
            nn.Linear(4*n_embd, n_embd),
            nn.Dropout(dropout)
        )
    
    def forward(self, xb):
        return self.ff(xb)

In [71]:
class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        # self.full_attend = MultipleSelfAttention(n_head, n_embd)
        self.comp_full_attend = CompleteSelfAttention(n_embd, n_head)
        self.ff = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, xb):
        attention = xb + self.comp_full_attend(self.ln1(xb))                      # (B, T, C) => 4 * self_attend(B, T, C/4) => (B, T, C)
        feed_forward = attention + self.ff(self.ln2(attention))

        return feed_forward

In [72]:
class LTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, n_embd)         # a token embedding allows us to represent letters as vectors
        self.pos_embedding = nn.Embedding(block_size, n_embd)           # attention also depends on the location of a token
                                                                        # the token embedding itself will give the same value to a character at the front vs at the end
                                                                        # pos embedding handles this
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for i in range(n_layers)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.last_layer = nn.Linear(n_embd, vocab_size)

    def forward(self, xb, targets):
        B, T = xb.shape

        tok_embd = self.token_embedding(xb)                             # (B, T, C)
        pos_embd = self.pos_embedding(torch.arange(T, device=xb.device))  # (T, C), positions built on the input's device
        x = tok_embd + pos_embd                                       # (B, T, C) + (T, C) => (B, T, C) + (B, T, C)

        out = self.blocks(x)
        logits = self.last_layer(self.ln_f(out))
        
        if targets is None: 
            loss = None
        else:
            flat_logits = logits.view(-1, vocab_size)
            flat_targets = targets.view(-1)
            loss = F.cross_entropy(flat_logits, flat_targets)
        return logits, loss
    
    def generate(self, tokens, idx):

        for i in range(tokens):
            logits, loss = self(idx[:, -block_size:], None)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            next_tok = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, next_tok), dim=1)
        
        return idx


xb, yb = generate_batch()

model = LTransformer().to(device)                                   # move all model parameters onto the active device
logits, loss = model(xb, yb)
print(logits, loss)

tensor([[[ 0.0961, -0.2199,  0.3974,  ..., -0.0103,  0.3346,  0.5757],
         [-0.3766,  1.0700,  0.9775,  ..., -0.8296,  0.0908,  0.0134],
         [ 0.7159, -0.3430, -0.0995,  ..., -1.6002, -0.1109, -0.4568],
         ...,
         [ 0.2296,  0.4803,  0.6073,  ...,  0.1484, -0.3145,  0.1383],
         [ 0.3638,  0.0963,  0.3252,  ...,  1.0468, -0.2639,  0.7330],
         [-0.0752, -0.4706, -0.1274,  ..., -0.7778, -0.2254,  0.4128]],

        [[-0.2663,  0.6339,  0.6320,  ..., -0.4547,  0.0636, -0.0602],
         [-0.9247,  0.9908,  0.3507,  ..., -0.2055,  0.3801,  0.4022],
         [ 0.0847,  0.0975, -0.7830,  ..., -1.1587,  0.1481, -0.5576],
         ...,
         [ 0.3914,  0.5253,  0.7496,  ..., -0.1310, -0.3205,  0.2908],
         [ 1.1662,  0.1846,  1.1178,  ..., -0.6994, -0.2725, -0.0475],
         [-0.2245, -0.4525, -0.6682,  ..., -0.5906, -0.2316, -0.3095]],

        [[ 0.0821, -0.6091,  0.6616,  ...,  0.8493,  0.1232,  1.2745],
         [-0.0377,  0.8153,  0.5311,  ..., -1

In [73]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

@torch.no_grad()
def estimate_loss(iters):
    out = {}
    model.eval()
    for split in ['train', 'test']:
        losses = torch.zeros(iters)
        for k in range(iters):
            xb, yb = generate_batch(split)                  # use the split's data (train vs test)
            tlogits, tloss = model(xb, yb)
            losses[k] = tloss.item()
        out[split] = losses.mean()
    model.train()
    return out

for i in range(2000):
    xb, yb = generate_batch()
    logits, loss = model(xb, yb)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if i % 1000 ==0:
        estimated_loss = estimate_loss(20)
        print(f"loss at epoch {i}: {estimated_loss['train']:.4f}, {estimated_loss['test']:.4f}")

loss at epoch 0: 5.2213, 5.2261
loss at epoch 1000: 2.2919, 2.3091


In [243]:
idx = torch.zeros((1, 1), dtype=torch.long, device=device)          # seed token, on the active device
out = model.generate(200, idx)
print(decode(out[0].tolist()))


between the essentagences of superful phenomena actions, “i happened
Zarathustra,” in all the functions of “great principle!”





248.



THE HONEST TASTE OF EDYMPLE.—That take cults it compression b
